In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [2]:
diabetes = pd.read_csv('diabetes_prevalence.csv')
food = pd.read_csv('food_access.csv')

In [3]:
diabetes.head()

,Year,StateAbbr,StateDesc,CountyName,CountyFIPS,LocationName,DataSource,Category,Measure,Data_Value_Unit,...,Data_Value_Footnote,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,Geolocation,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text
0,2018,AL,Alabama,Baldwin,1003,1003011601,BRFSS,Prevention,Cervical cancer screening among adult women ag...,%,...,NaN,83.5,87.1,"6,062",POINT (-87.55879555 30.41466842),1003011601,PREVENT,CERVICAL,CrdPrv,Cervical Cancer Screening
1,2019,ME,Maine,Waldo,23027,23027045000,BRFSS,Health Status,Fair or poor self-rated health status among ad...,%,...,NaN,17.4,20.3,"5,969",POINT (-69.23270168 44.42005508),23027045000,HLTHSTAT,GHLTH,CrdPrv,General Health
2,2019,AL,Alabama,Calhoun,1015,1015002300,BRFSS,Health Outcomes,Obesity among adults aged >=18 years,%,...,NaN,38.0,40.0,"3,843",POINT (-85.67101785 33.93457433),1015002300,HLTHOUT,OBESITY,CrdPrv,Obesity
3,2019,LA,Louisiana,Tangipahoa,22105,22105953500,BRFSS,Health Risk Behaviors,Current smoking among adults aged >=18 years,%,...,NaN,22.3,25.4,"7,225",POINT (-90.36823496 30.77718215),22105953500,RISKBEH,CSMOKING,CrdPrv,Current Smoking
4,2019,ME,Maine,Cumberland,23005,23005014000,BRFSS,Health Risk Behaviors,Current smoking among adults aged >=18 years,%,...,NaN,14.7,20.3,"3,872",POINT (-70.61427542 43.97054122),23005014000,RISKBEH,CSMOKING,CrdPrv,Current Smoking


In [4]:
food.head()

,CensusTract,State,County,Urban,Pop2010,OHU2010,GroupQuartersFlag,NUMGQTRS,PCTGQTRS,LILATracts_1And10,...,TractSeniors,TractWhite,TractBlack,TractAsian,TractNHOPI,TractAIAN,TractOMultir,TractHispanic,TractHUNV,TractSNAP
0,1001020100,Alabama,Autauga County,1,1912,693,0,0.0,0.00,0,...,221.0,1622.0,217.0,14.0,0.0,14.0,45.0,44.0,6.0,102.0
1,1001020200,Alabama,Autauga County,1,2170,743,0,181.0,8.34,1,...,214.0,888.0,1217.0,5.0,0.0,5.0,55.0,75.0,89.0,156.0
2,1001020300,Alabama,Autauga County,1,3373,1256,0,0.0,0.00,0,...,439.0,2576.0,647.0,17.0,5.0,11.0,117.0,87.0,99.0,172.0
3,1001020400,Alabama,Autauga County,1,4386,1722,0,0.0,0.00,0,...,904.0,4086.0,193.0,18.0,4.0,11.0,74.0,85.0,21.0,98.0
4,1001020500,Alabama,Autauga County,1,10766,4082,0,181.0,1.68,0,...,1126.0,8666.0,1437.0,296.0,9.0,48.0,310.0,355.0,230.0,339.0


In [5]:
diabetes = diabetes[diabetes['Measure'].str.contains('diabetes')] # Filters to only diabetes cases
diabetes = diabetes[['LocationName', 'Data_Value']]
diabetes.head()

,LocationName,Data_Value
11,23025966600,12.2
13,23031034002,8.7
50,23025965600,12.8
67,1083020300,10.6
72,1087231900,18.1


In [6]:
# Select Features
food = food[['CensusTract', 'State', 'County', 'LATracts_half', 'LATracts1', 
             'LATracts10', 'LATracts20', 'Pop2010', 'Urban', 'PCTGQTRS', 'PovertyRate',
             'MedianFamilyIncome', 'TractSeniors', 'TractWhite', 'TractBlack',
             'TractAsian', 'TractNHOPI', 'TractAIAN', 'TractHispanic']]

In [7]:
# Turns raw counts into rates
food['TractSeniorsRate'] = food['TractSeniors'] / food['Pop2010']
food['TractWhiteRate'] = food['TractWhite'] / food['Pop2010']
food['TractBlackRate'] = food['TractBlack'] / food['Pop2010']
food['TractAsianRate'] = food['TractAsian'] / food['Pop2010']
food['TractNHOPIRate'] = food['TractNHOPI'] / food['Pop2010']
food['TractAIANRate'] = food['TractAIAN'] / food['Pop2010']
food['TractHispanicRate'] = food['TractHispanic'] / food['Pop2010']
food['lapop10%'] = food['lapop10share'] / 100

# Turns percentages into proportions
food['PovertyRate%'] = food['PovertyRate'] / 100
food['GroupQuarters%'] = food['PCTGQTRS'] / 100

# Drops raw count columns
food = food.drop(['TractSeniors', 'TractWhite', 'TractBlack', 'TractAsian', 'TractNHOPI', 'TractAIAN',
           'TractHispanic', 'PovertyRate', 'PCTGQTRS'], axis = 1) 

KeyError: 'lapop10share'

In [ ]:
food.describe()

In [ ]:
# Turns diabetes percentage into a rate
diabetes['DiabetesRate'] = diabetes['Data_Value'] / 100
diabetes = diabetes.drop('Data_Value', axis = 1)

In [ ]:
diabetes.describe()

In [ ]:
# Dataset shapes
print(diabetes.shape)
print(food.shape)

In [ ]:
# Combines Datasets
df = diabetes.merge(food, left_on = 'LocationName', right_on = 'CensusTract', how = 'inner')

# Drop one tract ID column
df = df.drop('CensusTract', axis = 1)
df.head()

In [ ]:
# Create final food access feature
df.insert(2, 'FoodAccessScore', np.select(
    [
        df['LATracts20'] == 1,
        df['LATracts10'] == 1,
        df['LATracts1'] == 1,
        df['LATracts_half'] == 1
    ],
    [
        4, 
        3, 
        2, 
        1, 
    ], default = 0
))

# Drop original food access columns
df = df.drop(['LATracts_half', 'LATracts1', 'LATracts10', 'LATracts20'], axis = 1)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
X = df[['FoodAccessScore]]
y = df['DiabetesRate']

data = pd.concat([X, y], axis=1).dropna()

X = data[['FoodAccessScore']]
y = data['DiabetesRate']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

mdl = LinearRegression()
mdl.fit(X_train, y_train)
mdl.score(X_test, y_test)

In [ ]:
plt.scatter(X, y)
plt.xlabel('Food Access Score')
plt.ylabel('Diabetes Rate')


In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    'DiabetesRate ~ FoodAccessScore + Urban + MedianFamilyIncome + TractSeniorsRate',
    data=df
).fit()

print(model.summary())

In [ ]:
model = smf.ols(
    'DiabetesRate ~ FoodAccessScore * Urban + '
    'FoodAccessScore * MedianFamilyIncome + '
    'FoodAccessScore * TractSeniorsRate',
    data=df
).fit()

print(model.summary())